https://github.com/miladlink/TinyYoloV2

https://github.com/eriklindernoren/PyTorch-YOLOv3


# Setup (Only use the first time or if the dataset was changed)

In [1]:
# !python filter_sample_json.py

# Libraries

In [2]:
import os
# import time
# from PIL import Image
# import numpy as np
import json
import cv2
from tqdm import tqdm
# import skimage.io as io
# import matplotlib.pyplot as plt
# from pycocotools.coco import COCO
# import torch
import torch.optim as optim
# import torchvision
# from torchvision import transforms
import torchvision.transforms as transforms
from torchvision.datasets.coco import CocoDetection
from torch.utils.data import DataLoader

from utils.YOLOv2 import *
from models.YOLOv3 import load_model
from attacks.FGSM import FGSM
from attacks.PGD import PGD
from attacks.CW import CW
from attacks.noise import Noise
from detect import detect_image
from utils.loss import compute_loss
from utils.utils import load_classes, rescale_boxes, non_max_suppression, print_environment_info
from utils.augmentations import TRANSFORM_TRAIN, TRANSFORM_VAL
from utils.transforms import DEFAULT_TRANSFORMS, Resize, ResizeEval

# Helper functions + vars


In [3]:
def xyxy2xywh(x):
    # Convert nx4 boxes from [x1, y1, x2, y2] to [x, y, w, h] where xy1=top-left, xy2=bottom-right
    y = x.clone() if isinstance(x, torch.Tensor) else np.copy(x)
    y[..., 0] = (x[..., 0] + x[..., 2]) / 2  # x center
    y[..., 1] = (x[..., 1] + x[..., 3]) / 2  # y center
    y[..., 2] = x[..., 2] - x[..., 0]  # width
    y[..., 3] = x[..., 3] - x[..., 1]  # height
    return y

def xywh2xyxy(x):
    # Convert nx4 boxes from [x, y, w, h] to [x1, y1, x2, y2] where xy1=top-left, xy2=bottom-right
    y = x.clone() if isinstance(x, torch.Tensor) else np.copy(x)
    y[..., 0] = x[..., 0] - x[..., 2] / 2  # top left x
    y[..., 1] = x[..., 1] - x[..., 3] / 2  # top left y
    y[..., 2] = x[..., 0] + x[..., 2] / 2  # bottom right x
    y[..., 3] = x[..., 1] + x[..., 3] / 2  # bottom right y
    return y

def yolo2json(boxes, img_copy, image_id):
    # * put into coco format of x_min,y_min, width, height, bbox_conf, cls
    # yolo format is x_center, y_center, w, h, bbox_conf, cls_conf, cls
    predictions = []
    for box in boxes:
        x_center, y_center, w, h, conf, cls = box
        x_min = max(0, (x_center - w / 2) * img_copy.shape[3])
        y_min = max(0, (y_center - h / 2) * img_copy.shape[2])
        width = min(img_copy.shape[3], w * img_copy.shape[3])
        height = min(img_copy.shape[2], h * img_copy.shape[2])
        # print(x_min,y_min, width, height, bbox_conf, cls)
        predictions.append({
            'image_id': image_id,
            'category_id': int(id_list[int(cls)]) if modelv == 3 else int(cls),
            'bbox': [int(x_min), int(y_min), int(width), int(height)],
            'score': round(float(conf),2)
        })
    return predictions

def nms2yolo(boxes, img_copy):
    boxes = xyxy2xywh(boxes) # convert from coco to yolo: nms returns nx6 (x1, y1, x2, y2, conf, cls), change to center coordinates [x_center, y_center, width, height]
    boxes[:,0] = boxes[:,0]/img_copy.shape[3]
    boxes[:,1] = boxes[:,1]/img_copy.shape[2]
    boxes[:,2] = boxes[:,2]/img_copy.shape[3]
    boxes[:,3] = boxes[:,3]/img_copy.shape[2]
    return boxes

def saveImageWithBoxes(images, boxes, class_names, fileName):
    to_pil = transforms.ToPILImage()
    pil_image = to_pil(images.squeeze())
    pred_img = plot_boxes(pil_image, boxes, None, class_names)
    pred_img.save(fileName)

def saveImage(img):
    # * just for sanity check, output image. put the dim 3 at the back
    imageN = img.clone().detach()
    imageN = imageN.cpu().squeeze().permute(1, 2, 0).numpy()
    imageN = cv2.cvtColor(imageN, cv2.COLOR_RGB2BGR)
    # print(imageN.shape)
    cv2.imwrite("data/results/mygraph.jpg", imageN*255)

def getOneIter(dataloader):
    images, annotations = next(iter(dataloader))
    np.set_printoptions(linewidth=500)
    np.set_printoptions(suppress=True)
    print("dataloader out")
    print(annotations[0].numpy())


def imgToGreyscale(img):
    if img.shape[0] != 3:
        raise ValueError("Input tensor must have shape [3, H, W].")
    grayscale = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    grayscale_tensor = grayscale.unsqueeze(0).repeat(3, 1, 1)
    return grayscale_tensor

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

os.environ['CUDA_LAUNCH_BLOCKING'] = '1' # reset CUDA debugging environment variable
os.environ['TORCH_USE_CUDA_DSA'] = '1' # enable CUDA DSA for debugging

In [5]:
epochs = 50 # currently, 100 seems like it works very well
checkpoint_interval = epochs 
# if time is limited then make this smaller, do note that checkpoints are around 270MB per.
modelv = 3
img_size=416

In [6]:
class_names = ['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
id_list = np.array(range(0,80))

# Model import

In [7]:
# if modelv == 2:
#     model = load_model_v2(weights = './weights/yolov2-tiny-voc.weights').to(device)
#     class_names = ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'TVmonitor']
#     root_train = "./data/VOC2007/JPEGImages"
#     annFile_train = "./data/VOC2007/annotations/train.json"
#     root_val = "./data/VOC2007/JPEGImages"
#     annFile_val = "./data/VOC2007/annotations/val.json"

if modelv == 3:
    model = load_model("./config/yolov3.cfg", "./weights/yolov3.weights")
    root_train = "./data/COCO2017/images/train_sample"
    annFile_train = "./data/COCO2017/annotations/instances_train2017_modified_sample.json"
    root_val = "./data/COCO2017/images/valid_sample"
    annFile_val = "./data/COCO2017/annotations/instances_val2017_modified_sample.json"

else:
    print("invalid model number!")

# COCO loader

create dataloader (make different train and val later)

In [8]:
# coco_dataset_train = CocoDetection(root=root_train, annFile=annFile_train, transform=TRANSFORM_TRAIN_IMG, target_transform=TRANSFORM_TRAIN_TARGET)
# coco_dataset_train = CocoDetection(root=root_train, annFile=annFile_train, transforms=TRANSFORM_TRAIN)
coco_dataset_train = CocoDetection(root=root_val, annFile=annFile_val, transforms=TRANSFORM_TRAIN)
coco_dataset_val = CocoDetection(root=root_val, annFile=annFile_val, transforms=TRANSFORM_VAL)
# coco_dataset_eval = CocoDetection(root=root_val, annFile=annFile_val, transform=transforms.Compose([transforms.ToTensor(),]))

def collate_fn(batch):
    return tuple(zip(*batch))

# Create a DataLoader for your COCO dataset
train_loader = DataLoader(coco_dataset_val, batch_size=8, shuffle=True, collate_fn=collate_fn) # multiple images per batch
val_loader = DataLoader(coco_dataset_val, batch_size=1, shuffle=True, collate_fn=collate_fn)
# one per batch
# cocoeval_loader = DataLoader(coco_dataset_eval, batch_size=1, shuffle=True, collate_fn=collate_fn) # original images without transformatios


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [9]:
getOneIter(val_loader) # print targets

dataloader out
[[872.          32.           0.65160937   0.26881249   0.03028126   0.02582815]
 [872.           0.           0.24103124   0.15729687   0.45617187   0.71460941]
 [872.           0.           0.26989062   0.19753125   0.41514065   0.75062501]
 [872.          35.           0.59006252   0.24570312   0.0897656    0.07153125]]


# Adversarial training

In [10]:
eps = 0.05
# attacker = FGSM(model=model, epsilon=0.05)
# attacker = PGD(model=model, epsilon=0.05, epoch=5, lr=0.02)
attacker = CW(model=model, epsilon=eps, lr=eps/3, epoch=5, target=52) # 52 is banana
# attacker = Noise(model=model, epsilon=0.1)


In [11]:
losses = []
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(
            params,
            lr=model.hyperparams['learning_rate'],
            weight_decay=model.hyperparams['decay'],
        )

for epoch in range(1, epochs+1):
    print(f"Starting epoch {epoch}")
    lossesEpoch = []

    for batch_idx, (images, targets) in enumerate(tqdm(train_loader)):
        
        model.train()

        if targets[0].numel() != 0:
            try:
                #* modify inputs to be in proper shape
                images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
                images = images.to(device)

                # modify targets to be in proper shape
                for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
                    if boxes.ndim == 2:
                        boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss

                targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
                targets = targets[:, :6]
                # class indices are mixed up prior

                # # verify class indices range
                # class_indices = targets[:, 1].long()
                # valid_classes = (class_indices >= 0) & (class_indices < 80)

                # if not valid_classes.all():
                #     print(f"Warning: Invalid class indices found: {class_indices[~valid_classes]}")
                #     # Filter out invalid classes
                #     targets = targets[valid_classes]
                #     if targets.shape[0] == 0:
                #         print("No valid targets after filtering, skipping batch")
                #         continue

                # ensure all class indices are long
                targets[:, 1] = targets[:, 1].long()

                print(f"Batch {batch_idx}: targets shape: {targets.shape}, class range: {targets[:, 1].min()}-{targets[:, 1].max()}")

                images_adv = attacker.forward(images, targets) # get adversarial image
                outputsBefore = model(images)
                lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
                outputsAfter = model(images_adv)
                lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
                loss = lossBefore + lossAfter

                lossesEpoch.append(loss.detach().cpu().numpy())
                loss.backward()
                optimizer.step()
                # Reset gradients
                optimizer.zero_grad()

                time.sleep(0.1) # for using noise attack

            except RuntimeError as e:
                print(f"Error in batch {batch_idx}: {e}")
                print(f"Targets shape: {targets.shape if 'targets' in locals() else 'undefined'}")
                if 'targets' in locals():
                    print(f"Class indices: {targets[:, 1].unique()}")
                # clear gradients and continue to next batch
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                continue

        else:
            continue # pics without targets

    if lossesEpoch:
        losses_avg = np.average(lossesEpoch)
        print(f"Epoch {epoch} average loss: {losses_avg}")
        losses.append(losses_avg)

    if epoch % checkpoint_interval == 0:
        checkpoint_path = f"./data/results/checkpoints/yolov3_ckpt_{epoch}.pth"
        print(f"---- Saving checkpoint to: '{checkpoint_path}' ----")
        os.makedirs("./data/results/checkpoints", exist_ok=True)
        torch.save(model.state_dict(), checkpoint_path)

Starting epoch 1


  0%|          | 0/5 [00:00<?, ?it/s]

Batch 0: targets shape: torch.Size([60, 6]), class range: 0.0-75.0


 20%|██        | 1/5 [00:01<00:04,  1.18s/it]

Batch 1: targets shape: torch.Size([20, 6]), class range: 0.0-77.0


...

Batch 3: targets shape: torch.Size([18, 6]), class range: 0.0-74.0


 80%|████████  | 4/5 [00:02<00:00,  1.77it/s]

Batch 4: targets shape: torch.Size([30, 6]), class range: 0.0-77.0


100%|██████████| 5/5 [00:02<00:00,  1.77it/s]


Epoch 50 average loss: 0.12992040812969208
---- Saving checkpoint to: './data/results/checkpoints/yolov3_ckpt_50.pth' ----


## Load adversarial trained model

In [12]:
if modelv == 3:
    model = load_model("./config/yolov3.cfg", f"./data/results/checkpoints/yolov3_ckpt_{epochs}.pth") # <- this assumes checkpoint interval is a factor of epochs
    # class_names = ['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
    # id_list = np.array(range(0,90))
    root_train = "./data/COCO2017/images/train_sample"
    annFile_train = "./data/COCO2017/annotations/instances_train2017_modified_sample.json"
    root_val = "./data/COCO2017/images/valid_sample"
    annFile_val = "./data/COCO2017/annotations/instances_val2017_modified_sample.json"

else:
    print("invalid model number!")

# Attack Evaluation

In [13]:
# attackImage = 0 # variable for saving attack image, run this first, change pruning ratio (attack), 
# #don't run this and only run below cells

### NOTE: Attacker was defined here.

In [14]:
predictionsBefore = []
predictionsAfter = []
lossesBefore = []
lossesAfter = []
mode = "image" # need different modes if i want to save image or output prediction json
# mode = "json"
image_ids= [139, 285, 632, 724, 776, 785, 802, 872, 885, 1000,
            1268, 1296, 1353,1425, 1490, 1503, 1532, 1584, 1675, 1761]

os.makedirs("./data/results/images", exist_ok=True)

for i, (images, targets) in enumerate(tqdm(val_loader)):
    if targets[0].numel() != 0:
        with torch.no_grad():
            #* modify inputs to be in proper shape
            images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
            images = images.to(device)
            image_id = int(targets[0][0,0].cpu().numpy()) # assume 1 image
            if image_id not in image_ids: continue # for when we want outputs of specific images
            for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
                if boxes.ndim == 2: boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss. this is normally done in ListDataset -> collate_fn. the id now starts at 0 for each image
            targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
            # originalImageSize = targets[0, 6:].cpu().numpy() # original image shape, assume one image per batch - NOT available in json format
            img_info = coco_dataset_val.coco.imgs[image_id]
            originalImageSize = (img_info['height'], img_info['width'])
            targets = targets[:, :6]

            # print debugging information
            print(f"Image ID: {image_id}")
            print(f"Original targets shape: {targets.shape}")
            print(f"Targets data:")
            print(targets)
            print(f"Class indices: {targets[:, 1]}")
            print(f"Class range: {targets[:, 1].min()} - {targets[:, 1].max()}")

            # check mapping of class indices
            original_classes = targets[:, 1].clone()
            print(f"Original class IDs: {original_classes}")

            # mapping class indices ([1, 80] to [0, 79] range
            targets[:, 1] = targets[:, 1] - 1

            # verify mapped class indices
            mapped_classes = targets[:, 1]
            print(f"Mapped class IDs: {mapped_classes}")
            print(f"Mapped class range: {mapped_classes.min()} - {mapped_classes.max()}")

            # # ensure all classes are in range [0, 79]
            # valid_mask = (mapped_classes >= 0) & (mapped_classes < 80)
            # if not valid_mask.all():
            #     print(f"Invalid class indices found: {mapped_classes[~valid_mask]}")
            #     targets = targets[valid_mask]
            #     if targets.shape[0] == 0:
            #         print("No valid targets after filtering, skipping image")
            #         continue
            #     print(f"Filtered targets shape: {targets.shape}")

            # ensure all class indices are long
            targets[:, 1] = targets[:, 1].long()

            # final validation
            final_classes = targets[:, 1]
            print(f"Final class indices: {final_classes}")
            print(f"Final class range: {final_classes.min()} - {final_classes.max()}")
            print(f"All classes in range [0, 79]: {((final_classes >= 0) & (final_classes < 80)).all()}")

            #* loss
            model.train()
            try:
                outputsBefore = model(images)
                print(f"Model output shapes: {[out.shape for out in outputsBefore]}")

                lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
                lossesBefore.append(lossBefore.cpu().numpy())

                images_adv = attacker.forward(images, targets) # get adversarial image

                outputsAfter = model(images_adv)
                lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
                lossesAfter.append(lossAfter.cpu().numpy())

            except RuntimeError as e:
                print(f"CUDA error occurred: {e}")
                print(f"Error details:")
                print(f"  Targets shape: {targets.shape}")
                print(f"  Class indices: {targets[:, 1]}")
                # print(f"  Class unique values: {targets[:, 1].unique()}")
                print(f"  Class data type: {targets[:, 1].dtype}")

                # clean CUDA cache and skip this iteration
                torch.cuda.empty_cache()
                continue

            #* plot
            model.eval()

            # before attack
            outputsBefore = model(images[0].unsqueeze(0))
            boxesBefore = non_max_suppression(outputsBefore, conf_thres=0.3, iou_thres=0.5)[0].numpy()
            if mode == "json":
                boxesBefore = rescale_boxes(boxesBefore, img_size, originalImageSize)
            boxesBefore = nms2yolo(boxesBefore, images)
            if mode == "image":
                saveImageWithBoxes(images[0], boxesBefore, class_names, f"./data/results/images/attack_before_{image_id}.jpg")
            if mode == "json":
                predictionsBefore += yolo2json(boxesBefore, images[0].unsqueeze(0), image_id)

            # after attack
            outputsAfter = model(images_adv[0].unsqueeze(0))
            boxesAfter = non_max_suppression(outputsAfter, conf_thres=0.3, iou_thres=0.5)[0].numpy()

            if mode == "json":
                boxesAfter = rescale_boxes(boxesAfter, img_size, originalImageSize)
            boxesAfter = nms2yolo(boxesAfter, images_adv)
            print(boxesAfter)
            if mode == "image":
                saveImageWithBoxes(images_adv[0], boxesAfter, class_names, f"./data/results/images/attack_after_{image_id}.jpg")
            if mode == "json":
                predictionsAfter += yolo2json(boxesAfter, images_adv[0].unsqueeze(0), image_id)

    else: continue # pics without targets

with open(f'./data/results/predictionsBefore.json', 'w') as f:
    json.dump(predictionsBefore, f)
with open(f'./data/results/predictionsAfter.json', 'w') as f:
    json.dump(predictionsAfter, f)
np.savetxt("./data/results/lossesBefore.csv", lossesBefore, delimiter=",")
np.savetxt("./data/results/lossesAfter.csv", lossesAfter, delimiter=",")

  0%|          | 0/20 [00:00<?, ?it/s]

Image ID: 632
Original targets shape: torch.Size([18, 6])
Targets data:
tensor([[0.0000e+00, 5.9000e+01, 5.1094e-03, 5.3883e-01, 6.2692e-01, 3.2539e-01],
        [0.0000e+00, 5.8000e+01, 2.8650e-01, 3.3525e-01, 9.4969e-02, 1.4436e-01],
        [0.0000e+00, 7.3000e+01, 7.1247e-01, 4.2266e-01, 1.3391e-02, 5.5609e-02],
        [0.0000e+00, 7.3000e+01, 7.0830e-01, 5.1714e-01, 1.2531e-02, 5.3016e-02],
        [0.0000e+00, 7.3000e+01, 6.9494e-01, 5.8692e-01, 8.3125e-03, 6.2000e-02],
        [0.0000e+00, 7.3000e+01, 7.9055e-01, 4.2034e-01, 1.8984e-02, 5.7469e-02],
        [0.0000e+00, 7.3000e+01, 7.6173e-01, 4.3333e-01, 1.1687e-02, 4.3828e-02],
        [0.0000e+00, 5.6000e+01, 3.8253e-01, 4.8195e-01, 1.6363e-01, 1.3702e-01],
        [0.0000e+00, 5.8000e+01, 5.4273e-01, 4.5370e-01, 1.2892e-01, 2.2344e-01],
        [0.0000e+00, 7.3000e+01, 7.2017e-01, 4.2136e-01, 4.5828e-02, 5.6313e-02],
        [0.0000e+00, 7.3000e+01, 8.2347e-01, 5.1027e-01, 3.8125e-02, 6.3172e-02],
        [0.0000e+00, 7.300

  5%|▌         | 1/20 [00:00<00:08,  2.26it/s]

[[ 0.37976822  0.47996566  0.13524716  0.14167537  0.9421803  56.        ]
 [ 0.8248502   0.51418227  0.03856718  0.05951383  0.9231001  73.        ]
 [ 0.00950366  0.54514724  0.51569045  0.27680779  0.9089952  59.        ]
 [ 0.7193973   0.42094585  0.0404713   0.05487471  0.90326643 73.        ]
 [ 0.5455612   0.44884714  0.12155313  0.22413422  0.8282355  58.        ]
 [ 0.7898354   0.41942582  0.02557241  0.05730468  0.8028657  73.        ]
 [ 0.29206976  0.3338046   0.08850265  0.15719597  0.787224   58.        ]
 [ 0.64201844  0.1860417   0.20401548  0.4148546   0.5815011  73.        ]
 [ 0.7023448   0.42199665  0.02945445  0.05313287  0.54606456 73.        ]
 [ 0.70410055  0.51529145  0.01591066  0.04964359  0.34892783 73.        ]
 [ 0.7031663   0.49820217  0.03454707  0.04902737  0.34464204 73.        ]
 [ 0.7039118   0.43479064  0.01423014  0.04811874  0.30473477 73.        ]]
Image ID: 776
Original targets shape: torch.Size([4, 6])
Targets data:
tensor([[0.0000e+00, 7.7000e

 10%|█         | 2/20 [00:00<00:06,  2.85it/s]

...

[[0.19950148 0.13970383 0.5600002  0.7543317  0.940686   5.        ]
 [0.28154147 0.27954113 0.04164839 0.03529336 0.9142467  0.        ]
 [0.9288454  0.51400554 0.08641184 0.14145507 0.8833946  5.        ]
 [0.1443826  0.63565123 0.03659599 0.07468051 0.8775713  0.        ]
 [0.718267   0.54913545 0.02965957 0.04431072 0.8357621  0.        ]
 [0.8212233  0.5162297  0.1146431  0.11488907 0.82167745 5.        ]
 [0.3142555  0.49633646 0.09044354 0.10121456 0.7613961  0.        ]
 [0.6998554  0.5490173  0.03904431 0.04799689 0.52808005 0.        ]
 [0.18316863 0.64232147 0.04527921 0.08362656 0.48595884 0.        ]
 [0.294432   0.27926317 0.03902943 0.0325054  0.4440783  0.        ]
 [0.34900945 0.2776529  0.03553808 0.02926023 0.42488426 0.        ]
 [0.3112372  0.27850604 0.03909758 0.03056152 0.42045632 0.        ]
 [0.33363393 0.27723992 0.03959788 0.02853709 0.41133335 0.        ]
 [0.10271817 0.6050309  0.05448109 0.11646586 0.3455378  0.        ]
 [0.35423934 0.4947352  0.10774974

 95%|█████████▌| 19/20 [00:05<00:00,  3.30it/s]

[[ 0.09713034  0.39108866  0.55635124  0.3011656   0.9131742  48.        ]
 [ 0.759729    0.38759917  0.2172388   0.33945268  0.904367   45.        ]]
Image ID: 1761
Original targets shape: torch.Size([7, 6])
Targets data:
tensor([[0.0000, 4.0000, 0.6066, 0.2178, 0.0800, 0.0684],
        [0.0000, 4.0000, 0.4003, 0.0135, 0.1158, 0.1206],
        [0.0000, 0.0000, 0.1838, 0.9624, 0.0076, 0.0180],
        [0.0000, 0.0000, 0.1723, 0.9630, 0.0113, 0.0153],
        [0.0000, 0.0000, 0.1900, 0.9609, 0.0097, 0.0201],
        [0.0000, 0.0000, 0.2223, 0.9777, 0.0075, 0.0113],
        [0.0000, 0.0000, 0.1989, 0.9642, 0.0115, 0.0166]], device='cuda:0',
       dtype=torch.float64)
Class indices: tensor([4., 4., 0., 0., 0., 0., 0.], device='cuda:0', dtype=torch.float64)
Class range: 0.0 - 4.0
Original class IDs: tensor([4., 4., 0., 0., 0., 0., 0.], device='cuda:0', dtype=torch.float64)
Mapped class IDs: tensor([ 3.,  3., -1., -1., -1., -1., -1.], device='cuda:0',
       dtype=torch.float64)
Mapped cla

100%|██████████| 20/20 [00:06<00:00,  3.25it/s]

[[0.40255228 0.01569558 0.11315808 0.10740283 0.9258762  4.        ]
 [0.6072897  0.22203647 0.0883321  0.05923168 0.5765291  4.        ]
 [0.20052886 0.96454644 0.01223377 0.01354584 0.4014974  0.        ]
 [0.18328166 0.9641148  0.01267521 0.01316687 0.36110073 0.        ]]


In [15]:
# predictionsBefore = []
# predictionsAfter = []
# lossesBefore = []
# lossesAfter = []
# # mode = "image" # need different modes if i want to save image or output prediction json
# mode = "json"
# # image_ids= [71711,19221,22192] # output images that i want, 19221 is broccoli, 22192 is dog, 71711 is plane
# # image_ids= [139, 285, 632, 724, 776, 785, 802, 872, 885, 1000,
# #             1268, 1296, 1353,1425, 1490, 1503, 1532, 1584, 1675, 1761] # sample image id
# image_ids = [139]

# os.makedirs("./data/results/images", exist_ok=True)

# for i, (images, targets) in enumerate(tqdm(val_loader)):
#     if targets[0].numel() != 0:
#         with torch.no_grad():
#             #* modify inputs to be in proper shape
#             images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
#             images = images.to(device)
#             image_id = int(targets[0][0,0].cpu().numpy()) # assume 1 image
#             if image_id not in image_ids: continue # for when we want outputs of specific images
#             for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
#                 if boxes.ndim == 2: boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss. this is normally done in ListDataset -> collate_fn. the id now starts at 0 for each image
#             targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
#             # originalImageSize = targets[0, 6:].cpu().numpy() # original image shape, assume one image per batch - NOT available in json format
#             img_info = coco_dataset_val.coco.imgs[image_id]
#             originalImageSize = (img_info['height'], img_info['width'])
#             targets = targets[:, :6]

#             #* loss
#             model.train()
#             # start = time.time()
#             outputsBefore = model(images)
#             # end = time.time()
#             # print(end - start)
#             lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
#             lossesBefore.append(lossBefore.cpu().numpy())

#             images_adv = attacker.forward(images, targets) # get adversarial image

#             outputsAfter = model(images_adv)
#             lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
#             lossesAfter.append(lossAfter.cpu().numpy())

#             #* plot
#             model.eval()

#             # ground truth
#             # print(targets) #(ima ge,class,x,y,w,h), the class id starts from 1
#             # nms is (x1, y1, x2, y2, conf, cls), the class id starts from 0
#             # yolo is (x_center, y_center, width, height, conf. cls)

#             # before attack
#             outputsBefore = model(images[0].unsqueeze(0))
#             boxesBefore = non_max_suppression(outputsBefore, conf_thres=0.3, iou_thres=0.5)[0].numpy()
#             if mode == "json":
#                 boxesBefore = rescale_boxes(boxesBefore, img_size, originalImageSize)
#             boxesBefore = nms2yolo(boxesBefore, images)
#             if mode == "image":
#                 saveImageWithBoxes(images[0], boxesBefore, class_names, f"./data/results/images/attack_before_{image_id}.jpg")
#             if mode == "json":
#                 predictionsBefore += yolo2json(boxesBefore, images[0].unsqueeze(0), image_id)

#             # after attack
#             outputsAfter = model(images_adv[0].unsqueeze(0))
#             boxesAfter = non_max_suppression(outputsAfter, conf_thres=0.3, iou_thres=0.5)[0].numpy()


#             if mode == "json":
#                 boxesAfter = rescale_boxes(boxesAfter, img_size, originalImageSize)
#             # print(boxesAfter)
#             boxesAfter = nms2yolo(boxesAfter, images_adv)
#             print(boxesAfter)
#             if mode == "image":
#                 saveImageWithBoxes(images_adv[0], boxesAfter, class_names, f"./data/results/images/attack_after_{image_id}.jpg")

#                 # attackImage = images_adv[0] # for saving the same attack image for different pruning ratios, comment out after save
#                 # saveImageWithBoxes(attackImage, boxesAfter, class_names, f"./data/results/images/pruning/{image_id}/attack_after_99_x.jpg") # plot different pruning ratios with same attack image

#                 # greyscaleAttackImage = imgToGreyscale(attackImage)
#                 # saveImageWithBoxes(greyscaleAttackImage, boxesAfter, class_names, f"./data/results/images/pruning/{image_id}/attack_after_x_grey.jpg") # plot different pruning ratios with same attack image
#             if mode == "json":
#                 predictionsAfter += yolo2json(boxesAfter, images_adv[0].unsqueeze(0), image_id)
#             # time.sleep(0.1) # for using noise attack

#     else: continue # pics without targets
#     # break


# with open(f'./data/results/predictionsBefore.json', 'w') as f:
#     json.dump(predictionsBefore, f)
# with open(f'./data/results/predictionsAfter.json', 'w') as f:
#     json.dump(predictionsAfter, f)
# np.savetxt("./data/results/lossesBefore.csv", lossesBefore, delimiter=",")
# np.savetxt("./data/results/lossesAfter.csv", lossesAfter, delimiter=",")

In [16]:
# data = np.loadtxt('./data/results/lossesBefore.csv', delimiter=',')
# average = np.mean(data)
# print("Avg loss before attack:", average)
# data = np.loadtxt('./data/results/lossesAfter.csv', delimiter=',')
# average = np.mean(data)
# print("Avg loss after attack:", average)

# Get mAP

In [17]:
# from pycocotools.coco import COCO
# from pycocotools.cocoeval import COCOeval

# coco_gld = COCO(annFile_val) # coco
# # if modelv == 2:
# #     coco_rst = coco_gld.loadRes('./data/results/v2predictions.json')
# # elif modelv == 3:
# #     coco_rst = coco_gld.loadRes('./data/results/v3predictions.json')

# coco_rst = coco_gld.loadRes('./data/results/predictionsAfter.json')
# cocoEval = COCOeval(coco_gld, coco_rst, iouType='bbox')
# cocoEval.evaluate()
# cocoEval.accumulate()
# cocoEval.summarize()